# Task1 — 질문유형 분류기 추론
라벨: 졸업요건=0, 학교공지=1, 학사일정=2, 식단=3, 통학/셔틀=4

`model/` 에서 학습된 분류기를 로드해 `data/test_cls.json` 을 예측하고 `outputs/cls_output.json` (`[{"question":...,"label":N}]`) 으로 저장한다.

학습: `python scripts/train_classifier.py` (풀 학습은 GPU 권장).

In [ ]:
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ROOT: 노트북은 __file__ 이 없으므로 cwd 기준으로 repo 루트를 찾는다.
ROOT = Path.cwd()
if ROOT.name == 'src':
    ROOT = ROOT.parent
while not (ROOT / 'model').exists() and ROOT != ROOT.parent:
    if (ROOT / 'scripts' / 'train_classifier.py').exists():
        break
    ROOT = ROOT.parent

MODEL_DIR = ROOT / 'model'
TEST_PATH = ROOT / 'data' / 'test_cls.json'
VALID_PATH = ROOT / 'data' / 'cls' / 'valid.json'
OUT_DIR = ROOT / 'outputs'
OUT_PATH = OUT_DIR / 'cls_output.json'
MAX_LEN, BATCH = 64, 32
print('ROOT =', ROOT)

In [ ]:
# test_cls.json 이 없으면 valid.json 의 question 만 떼서 임시 생성(스모크)
if not TEST_PATH.exists():
    with open(VALID_PATH, encoding='utf-8') as f:
        rows = json.load(f)
    test_rows = [{'question': r['question']} for r in rows]
    TEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(TEST_PATH, 'w', encoding='utf-8') as f:
        json.dump(test_rows, f, ensure_ascii=False, indent=2)
    print('test_cls.json 임시 생성:', TEST_PATH, 'n =', len(test_rows))
else:
    print('test_cls.json 사용:', TEST_PATH)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device).eval()
print('device =', device, '| model_dir =', MODEL_DIR)

In [ ]:
with open(TEST_PATH, encoding='utf-8') as f:
    test_rows = json.load(f)
questions = [r['question'] for r in test_rows]

preds = []
with torch.no_grad():
    for i in range(0, len(questions), BATCH):
        batch = questions[i:i + BATCH]
        enc = tokenizer(batch, truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors='pt').to(device)
        logits = model(**enc).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

# 공식 템플릿 양식: id, question, label. id 는 입력행 id 있으면 사용, 없으면 인덱스.
out = [{'id': test_rows[i].get('id', i), 'question': q, 'label': int(p)}
       for i, (q, p) in enumerate(zip(questions, preds))]
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(len(out), '건 예측 →', OUT_PATH)
for row in out[:3]:
    print('  ', json.dumps(row, ensure_ascii=False))